**Create Table on BigQuery**

In [1]:
from google.cloud import bigquery
from google.oauth2 import service_account

# =========================================
# GOOGLE BIGQUERY AUTHENTICATION
# =========================================

SERVICE_ACCOUNT_FILE = "../Keys/BigQueryKey.json"

credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE
)

# =========================================
# CONFIGURATION
# =========================================

PROJECT_ID = "depihealthnux"
DATASET_ID = "Depihealthnux_Main"
TABLE_ID = "Labs_Ref"

TABLE_REF = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

# =========================================
# INITIALIZE BIGQUERY CLIENT
# =========================================

client = bigquery.Client(
    credentials=credentials,
    project=PROJECT_ID
)

# =========================================
# TABLE SCHEMA
# =========================================

schema = [
    bigquery.SchemaField(
        "Lab_Code",
        "STRING",
        mode="REQUIRED"
    ),

    bigquery.SchemaField(
        "Lab_Group",
        "STRING"
    ),

    bigquery.SchemaField(
        "Lab_Name",
        "STRING",
        mode="REQUIRED"
    )
]

# =========================================
# CREATE TABLE OBJECT
# =========================================

table = bigquery.Table(
    TABLE_REF,
    schema=schema
)

# =========================================
# CLUSTERING
# =========================================

table.clustering_fields = [
    "Lab_Group",
    "Lab_Code"
]

# =========================================
# CREATE TABLE
# =========================================

table = client.create_table(
    table,
    exists_ok=True
)

print("=================================")
print("Table created successfully")
print(TABLE_REF)
print("=================================")

Table created successfully
depihealthnux.Depihealthnux_Main.Labs_Ref


**Insert Data into BigQuery**

In [5]:
import pandas as pd

from google.cloud import bigquery
from google.oauth2 import service_account

# =========================================
# AUTHENTICATION
# =========================================

SERVICE_ACCOUNT_FILE = "../Keys/BigQueryKey.json"

credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE
)

client = bigquery.Client(
    credentials=credentials,
    project="depihealthnux"
)

# =========================================
# READ EXCEL
# =========================================

df = pd.read_excel("../Resources/Labs_Ref.xlsx")

print(df.head())

# =========================================
# CLEAN DATA
# =========================================

df = df.fillna("")

df["Lab_Code"] = df["Lab_Code"].astype(str).str.strip()
df["Lab_Group"] = df["Lab_Group"].astype(str).str.strip()
df["Lab_Name"] = df["Lab_Name"].astype(str).str.strip()

# =========================================
# LOAD TO BIGQUERY
# =========================================

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

job = client.load_table_from_dataframe(
    df,
    "depihealthnux.Depihealthnux_Main.Labs_Ref",
    job_config=job_config
)

job.result()

print(f"{len(df)} rows loaded successfully")

  Lab_Code      Lab_Group           Lab_Name
0    LC001     DM Profile              HbA1C
1    LC002  Lipid Profile                LDL
2    LC003  Lipid Profile                 TG
3    LC004  Lipid Profile  Total Cholesterol
4    LC005            LFT         SGPT (ALT)
27 rows loaded successfully


**Create Postgres Table**

In [3]:
import sys
import psycopg2

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

# =========================================
# CONNECT TO POSTGRES
# =========================================

conn = psycopg2.connect(POSTGRES_URL)
cursor = conn.cursor()

# =========================================
# CREATE SEQUENCE
# =========================================

cursor.execute("""

CREATE SEQUENCE IF NOT EXISTS lab_no_seq;

""")

# =========================================
# CREATE TABLE
# =========================================

create_table_query = """
CREATE TABLE IF NOT EXISTS Labs_Ref (

    Lab_Code TEXT PRIMARY KEY
    DEFAULT (
        'LC' ||
        LPAD(
            nextval('lab_no_seq')::TEXT,
            3,
            '0'
        )
    ),

    Lab_Group TEXT,
    Lab_Name TEXT NOT NULL

);
"""

cursor.execute(create_table_query)

conn.commit()

print("=================================")
print("PostgreSQL table created successfully")
print("Table: Labs_Ref")
print("=================================")

cursor.close()
conn.close()

PostgreSQL table created successfully
Table: Labs_Ref


**Test BigQuery to Postgres Sync**

In [6]:
import sys
import pandas as pd
import psycopg2

from psycopg2.extras import execute_values

from google.cloud import bigquery
from google.oauth2 import service_account

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

# =========================================
# BIGQUERY AUTH
# =========================================

credentials = service_account.Credentials.from_service_account_file(
    "../Keys/BigQueryKey.json"
)

client = bigquery.Client(
    credentials=credentials,
    project="depihealthnux"
)

# =========================================
# READ BIGQUERY
# =========================================

query = """
SELECT
    Lab_Code,
    Lab_Group,
    Lab_Name
FROM `depihealthnux.Depihealthnux_Main.Labs_Ref`
ORDER BY Lab_Code
"""

df = client.query(query).to_dataframe()

print(df.head(3))

# =========================================
# CONNECT POSTGRES
# =========================================

conn = psycopg2.connect(POSTGRES_URL)
cursor = conn.cursor()

cursor.execute("""
DELETE FROM Labs_Ref;
""")

if not df.empty:

    execute_values(
        cursor,
        """
        INSERT INTO Labs_Ref (
            Lab_Code,
            Lab_Group,
            Lab_Name
        )
        VALUES %s
        """,
        df.values.tolist()
    )

# =========================================
# RESET SEQUENCE
# =========================================

cursor.execute("""

SELECT setval(
    'lab_no_seq',
    COALESCE(
        (
            SELECT MAX(
                CAST(
                    REPLACE(Lab_Code,'LC','')
                    AS INTEGER
                )
            )
            FROM Labs_Ref
        ),
        1
    ),
    true
);

""")

conn.commit()

print(f"Inserted {len(df)} rows")

cursor.execute("""

SELECT
    Lab_Code,
    Lab_Group,
    Lab_Name
FROM Labs_Ref
ORDER BY Lab_Code
LIMIT 3

""")

print("\nFirst 3 Rows From PostgreSQL")

for row in cursor.fetchall():
    print(row)

cursor.close()
conn.close()

  Lab_Code      Lab_Group Lab_Name
0    LC001     DM Profile    HbA1C
1    LC002  Lipid Profile      LDL
2    LC003  Lipid Profile       TG
Inserted 27 rows

First 3 Rows From PostgreSQL
('LC001', 'DM Profile', 'HbA1C')
('LC002', 'Lipid Profile', 'LDL')
('LC003', 'Lipid Profile', 'TG')


**Test Postgres to BigQuery Sync**

In [7]:
import sys
import pandas as pd
import psycopg2

from google.cloud import bigquery
from google.oauth2 import service_account

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

# =========================================
# BIGQUERY AUTH
# =========================================

credentials = service_account.Credentials.from_service_account_file(
    "../Keys/BigQueryKey.json"
)

client = bigquery.Client(
    credentials=credentials,
    project="depihealthnux"
)

# =========================================
# READ POSTGRES
# =========================================

conn = psycopg2.connect(POSTGRES_URL)

query = """

SELECT
    Lab_Code,
    Lab_Group,
    Lab_Name
FROM Labs_Ref
ORDER BY Lab_Code

"""

df = pd.read_sql(query, conn)

print(df.head(3))

conn.close()

# =========================================
# LOAD TO BIGQUERY
# =========================================

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

job = client.load_table_from_dataframe(
    df,
    "depihealthnux.Depihealthnux_Main.Labs_Ref",
    job_config=job_config
)

job.result()

print(f"Loaded {len(df)} rows")

# =========================================
# VERIFY
# =========================================

verify_query = """

SELECT
    Lab_Code,
    Lab_Group,
    Lab_Name
FROM `depihealthnux.Depihealthnux_Main.Labs_Ref`
ORDER BY Lab_Code
LIMIT 3

"""

verify_df = client.query(
    verify_query
).to_dataframe()

print("\nFirst 3 Rows From BigQuery")

print(verify_df)

C:\Users\Sedeek Elmasry\AppData\Local\Temp\ipykernel_18296\2916001310.py:41: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


  lab_code      lab_group lab_name
0    LC001     DM Profile    HbA1C
1    LC002  Lipid Profile      LDL
2    LC003  Lipid Profile       TG
Loaded 27 rows

First 3 Rows From BigQuery
  Lab_Code      Lab_Group Lab_Name
0    LC001     DM Profile    HbA1C
1    LC002  Lipid Profile      LDL
2    LC003  Lipid Profile       TG
